# SuperKart: 2026 re-run with traceable model selection

**Author:** Mersedyes Livingston · re-run November 2026 · original project completed August 2025

The original modeling notebook was not recovered. The [2026 re-audit](../README.md) found that the presentation reports a random forest as the final model, while the deployed artifact contains XGBoost. This notebook rebuilds the model behind the deployed API's **exact 10-field input contract** and adds the provenance that would have caught the mismatch:

1. **Same features as the live API:** `POST /v1/predict` takes 10 fields; the pipeline consumes exactly those.
2. **CV-only selection:** selection is by 5-fold cross-validation on the training set (selection metric fixed in advance: RMSE). The test set is used once.
3. **Leave-one-store-out check:** with only 4 stores, does the model generalize to a store it has never seen?
4. **Model card with the artifact's SHA-256:** the evaluated artifact is identifiable, so a deployment can be verified against it.
5. **Payload validation:** the check the current API lacks (a missing field there raises an unhandled error).

**Data:** not redistributed. Put `SuperKart.csv` in `data/` at the repository root.


In [1]:
import hashlib, io, json, platform, tempfile, warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn, xgboost
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupKFold, KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", message="A worker stopped")
SEED = 1
DATA = Path("../../data/SuperKart.csv")
OUT_DIR = Path(".")
pd.set_option("display.float_format", "{:.3f}".format)
print(f"Python {platform.python_version()} | scikit-learn {sklearn.__version__} | xgboost {xgboost.__version__} | pandas {pd.__version__}")

Python 3.13.18 | scikit-learn 1.9.1 | xgboost 3.4.1 | pandas 3.0.5


## 1. Load and validate

In [2]:
REQUIRED = ["Product_Id", "Product_Weight", "Product_Sugar_Content", "Product_Allocated_Area", "Product_Type",
            "Product_MRP", "Store_Id", "Store_Establishment_Year", "Store_Size", "Store_Location_City_Type",
            "Store_Type", "Product_Store_Sales_Total"]
if not DATA.is_file():
    raise FileNotFoundError(f"Put SuperKart.csv at {DATA.resolve()} (data is not redistributed)")
raw = pd.read_csv(DATA)
missing = [c for c in REQUIRED if c not in raw.columns]
assert not missing, f"missing columns: {missing}"
assert raw[REQUIRED].notna().all().all(), "unexpected missing values"
DATA_SHA256 = hashlib.sha256(DATA.read_bytes()).hexdigest()
print(f"{len(raw):,} product-store rows | {raw.Store_Id.nunique()} stores | {raw.Product_Id.nunique():,} products")
print("sugar-content labels:", raw.Product_Sugar_Content.value_counts().to_dict())
print("data sha256:", DATA_SHA256)

8,763 product-store rows | 4 stores | 8,763 products
sugar-content labels: {'Low Sugar': 4885, 'Regular': 2251, 'No Sugar': 1519, 'reg': 108}
data sha256: b12ec41668e69d71d64ca96f2be78a011087f2a9cac6aba6e5cc920544b21ab2


**Data notes found in the re-audit:**
- `reg` and `Regular` are the same sugar-content category and are merged.
- Each of the 4 stores has a fixed size, city tier, and type, so the "store" features identify the store. A random split cannot show how the model would do at a new store (section 6).

In [3]:
raw.groupby("Store_Id")[["Store_Size", "Store_Location_City_Type", "Store_Type", "Store_Establishment_Year"]].agg(lambda s: s.unique().tolist())

,Store_Size,Store_Location_City_Type,Store_Type,Store_Establishment_Year
Store_Id,,,,
OUT001,[High],[Tier 2],[Supermarket Type1],[1987]
OUT002,[Small],[Tier 3],[Food Mart],[1998]
OUT003,[Medium],[Tier 1],[Departmental Store],[1999]
OUT004,[Medium],[Tier 2],[Supermarket Type2],[2009]


## 2. Features: the API's 10-field contract
The derived fields follow the 2025 feature engineering:
- `Store_Age_Years` = 2025 − establishment year.
- `Product_Id_char` = the product-family prefix (`FD` food, `DR` drinks, `NC` non-consumable).
- `Product_Type_Category` = 16 product types grouped into Food / Drinks / Household / Others.

The 2025 MRP capping is omitted. Tree models are unaffected by monotone caps, and leaving it out avoids a custom transformer inside the pickled pipeline. That transformer would have to be importable wherever the model is loaded, which is a hidden deployment dependency.

In [4]:
TYPE_GROUP = {**{t: "Food" for t in ["Fruits and Vegetables", "Snack Foods", "Frozen Foods", "Dairy", "Baking Goods",
                                      "Canned", "Meat", "Breads", "Starchy Foods", "Breakfast", "Seafood"]},
              "Soft Drinks": "Drinks", "Hard Drinks": "Drinks", "Household": "Household",
              "Health and Hygiene": "Household", "Others": "Others"}
API_FIELDS = {  # name -> type expected by POST /v1/predict
    "Product_Weight": float, "Product_Sugar_Content": str, "Product_Allocated_Area": float, "Product_MRP": float,
    "Store_Size": str, "Store_Location_City_Type": str, "Store_Type": str, "Product_Id_char": str,
    "Store_Age_Years": int, "Product_Type_Category": str}
NUMERIC = [k for k, t in API_FIELDS.items() if t in (int, float)]
CATEGORICAL = [k for k, t in API_FIELDS.items() if t is str]
TARGET = "Product_Store_Sales_Total"

def prepare(df):
    df = df.copy()
    df["Product_Sugar_Content"] = df["Product_Sugar_Content"].replace({"reg": "Regular"})
    df["Store_Age_Years"] = 2025 - df["Store_Establishment_Year"]
    df["Product_Id_char"] = df["Product_Id"].str[:2]
    df["Product_Type_Category"] = df["Product_Type"].map(TYPE_GROUP)
    assert df["Product_Type_Category"].notna().all(), "unmapped Product_Type"
    return df[list(API_FIELDS)], df[TARGET], df["Store_Id"]

X, y, stores = prepare(raw)
print(pd.crosstab(X.Product_Id_char, X.Product_Sugar_Content))

Product_Sugar_Content  Low Sugar  No Sugar  Regular
Product_Id_char                                    
DR                           498         0      207
FD                          4387         0     2152
NC                             0      1519        0


## 3. Split, candidates, cross-validation on the training set

In [5]:
X_tr, X_te, y_tr, y_te, s_tr, s_te = train_test_split(X, y, stores, test_size=0.30, random_state=SEED)
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
prep = lambda: ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL), ("num", "passthrough", NUMERIC)])
SCORING = {"rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error", "r2": "r2"}
CANDIDATES = {
    "Baseline: predict the mean": (DummyRegressor(), {}),
    "Linear regression": (LinearRegression(), {}),
    "Random forest": (RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1),
                      {"model__max_depth": [None, 12], "model__min_samples_leaf": [1, 5]}),
    "XGBoost": (XGBRegressor(n_estimators=400, random_state=SEED, n_jobs=-1),
                {"model__max_depth": [3, 5], "model__learning_rate": [0.05, 0.1]}),
}
fitted, rows = {}, []
for name, (est, grid) in CANDIDATES.items():
    pipe = Pipeline([("prep", prep()), ("model", est)])
    if grid:
        gs = GridSearchCV(pipe, grid, scoring="neg_root_mean_squared_error", cv=cv, n_jobs=-1).fit(X_tr, y_tr)
        best, params = gs.best_estimator_, {k.replace("model__", ""): v for k, v in gs.best_params_.items()}
    else:
        best, params = pipe.fit(X_tr, y_tr), {}
    s = cross_validate(best, X_tr, y_tr, cv=cv, scoring=SCORING, n_jobs=-1)
    fitted[name] = best
    rows.append({"model": name, "rmse": -s["test_rmse"].mean(), "rmse_sd": s["test_rmse"].std(),
                 "mae": -s["test_mae"].mean(), "r2": s["test_r2"].mean(), "params": params})
cv_table = pd.DataFrame(rows).set_index("model").sort_values("rmse")
cv_table

,rmse,rmse_sd,mae,r2,params
model,,,,,
Random forest,280.770,23.904,107.019,0.930,"{'max_depth': None, 'min_samples_leaf': 5}"
XGBoost,285.423,22.044,121.238,0.928,"{'learning_rate': 0.05, 'max_depth': 5}"
Linear regression,447.914,12.119,263.023,0.822,{}
Baseline: predict the mean,1063.715,17.295,838.177,-0.001,{}


## 4. Select by CV RMSE, then use the test set once

In [6]:
final_name = cv_table.drop(index="Baseline: predict the mean")["rmse"].idxmin()
final = fitted[final_name].fit(X_tr, y_tr)

def reg_metrics(y_true, pred):
    return {"rmse": float(np.sqrt(mean_squared_error(y_true, pred))), "mae": float(mean_absolute_error(y_true, pred)),
            "r2": float(r2_score(y_true, pred))}

def bootstrap(y_true, pred, n=2000, seed=SEED):
    rng, y_true, pred = np.random.default_rng(seed), np.asarray(y_true), np.asarray(pred)
    d = [reg_metrics(y_true[i], pred[i]) for i in (rng.integers(0, len(y_true), len(y_true)) for _ in range(n))]
    return {k: [float(np.percentile([x[k] for x in d], 2.5)), float(np.percentile([x[k] for x in d], 97.5))] for k in d[0]}

pred_te = final.predict(X_te)
test = {"Baseline: predict the mean": reg_metrics(y_te, fitted["Baseline: predict the mean"].predict(X_te)),
        final_name: reg_metrics(y_te, pred_te)}
test_ci = bootstrap(y_te, pred_te)
print(f"selected by CV: {final_name} {cv_table.loc[final_name, 'params']}")
display(pd.DataFrame(test).T)
print("95% bootstrap CI (selected model):", {k: f"[{a:.3f}, {b:.3f}]" for k, (a, b) in test_ci.items()})

selected by CV: Random forest {'max_depth': None, 'min_samples_leaf': 5}


,rmse,mae,r2
Baseline: predict the mean,1069.938,844.670,-0.000
Random forest,292.342,110.236,0.925


95% bootstrap CI (selected model): {'rmse': '[263.401, 321.745]', 'mae': '[100.349, 120.963]', 'r2': '[0.910, 0.939]'}


## 5. Which model was right? Documentation vs. deployment
- The 2025 presentation reported a tuned random forest as final.
- The deployed artifact (Hugging Face's scan of `final_model.joblib`) contains XGBoost.
- Below, both are compared under the same CV protocol, so the choice is made by evidence and recorded, not remembered.

In [7]:
cv_table.loc[["Random forest", "XGBoost"], ["rmse", "rmse_sd", "mae", "r2", "params"]]

,rmse,rmse_sd,mae,r2,params
model,,,,,
Random forest,280.770,23.904,107.019,0.930,"{'max_depth': None, 'min_samples_leaf': 5}"
XGBoost,285.423,22.044,121.238,0.928,"{'learning_rate': 0.05, 'max_depth': 5}"


## 6. Leave-one-store-out: does the model generalize to a new store?
The selected configuration is refit 4 times, each time holding out one entire store. Only 4 stores exist, so this is the strongest generalization test the data allows.

In [8]:
logo = GroupKFold(n_splits=stores.nunique())
lso_rows = []
for tr_idx, te_idx in logo.split(X, y, stores):
    held = stores.iloc[te_idx].iloc[0]
    m = Pipeline([("prep", prep()), ("model", final.named_steps["model"].__class__(**final.named_steps["model"].get_params()))])
    m.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    lso_rows.append({"held_out_store": held, "n": len(te_idx), **reg_metrics(y.iloc[te_idx], m.predict(X.iloc[te_idx]))})
lso = pd.DataFrame(lso_rows).set_index("held_out_store")
lso

,n,rmse,mae,r2
held_out_store,,,,
OUT004,4676,1739.884,1636.272,-12.808
OUT001,1586,1088.604,945.495,-0.449
OUT003,1349,1733.526,1565.667,-5.551
OUT002,1152,2603.523,2537.939,-30.666


## 7. Model card: make the evaluated artifact identifiable

In [9]:
with tempfile.TemporaryDirectory() as d:  # the artifact itself is not committed (pickles are gitignored)
    path = Path(d) / "final_model.joblib"
    joblib.dump(final, path)
    ARTIFACT_SHA256 = hashlib.sha256(path.read_bytes()).hexdigest()
    ARTIFACT_BYTES = path.stat().st_size

model_card = {
    "model_name": "superkart-sales-regressor", "version": "2026.11-rerun",
    "created_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    "estimator": type(final.named_steps["model"]).__name__, "hyperparameters": cv_table.loc[final_name, "params"],
    "artifact_sha256": ARTIFACT_SHA256, "artifact_bytes": ARTIFACT_BYTES,
    "training_data_sha256": DATA_SHA256, "n_train": int(len(X_tr)), "n_test": int(len(X_te)),
    "input_schema": {k: t.__name__ for k, t in API_FIELDS.items()}, "target": TARGET,
    "selection": "lowest 5-fold CV RMSE on training set",
    "metrics": {"cv": cv_table.loc[final_name, ["rmse", "rmse_sd", "mae", "r2"]].to_dict(),
                "test": test[final_name], "test_ci95": test_ci, "leave_one_store_out": lso.reset_index().to_dict(orient="records")},
    "limitations": ["Trained on 4 stores; see leave-one-store-out results before using for a new store.",
                    "Pickle artifact: load only if its SHA-256 matches this card."],
    "versions": {"python": platform.python_version(), "sklearn": sklearn.__version__, "xgboost": xgboost.__version__,
                 "pandas": pd.__version__, "numpy": np.__version__},
}
(OUT_DIR / "model_card.json").write_text(json.dumps(model_card, indent=2, default=float) + "\n")
print(f"estimator: {model_card['estimator']} | artifact sha256: {ARTIFACT_SHA256[:16]}… ({ARTIFACT_BYTES:,} bytes)")
print("wrote model_card.json. A deploy pipeline can now refuse any artifact whose hash differs.")

estimator: RandomForestRegressor | artifact sha256: 605be35ba3681ed4… (27,042,026 bytes)
wrote model_card.json. A deploy pipeline can now refuse any artifact whose hash differs.


## 8. Payload validation for `POST /v1/predict`
The deployed API reads `data['<field>']` directly, so a missing or mistyped field becomes an unhandled exception (HTTP 500). The function below is the contract the API should enforce: it returns field-level errors (HTTP 422) and only then calls the model. It is tested here against the trained pipeline.

In [10]:
def validate_payload(payload: dict) -> tuple[pd.DataFrame | None, list[str]]:
    errors = [f"missing field: {k}" for k in API_FIELDS if k not in payload]
    errors += [f"unexpected field: {k}" for k in payload if k not in API_FIELDS]
    for k, t in API_FIELDS.items():
        if k in payload and not (isinstance(payload[k], t) or (t is float and isinstance(payload[k], int))):
            errors.append(f"{k}: expected {t.__name__}, got {type(payload[k]).__name__}")
    return (None, errors) if errors else (pd.DataFrame([payload])[list(API_FIELDS)], [])

good = {"Product_Weight": 12.66, "Product_Sugar_Content": "Low Sugar", "Product_Allocated_Area": 0.027,
        "Product_MRP": 117.08, "Store_Size": "Medium", "Store_Location_City_Type": "Tier 2",
        "Store_Type": "Supermarket Type2", "Product_Id_char": "FD", "Store_Age_Years": 16,
        "Product_Type_Category": "Food"}
bad = {k: v for k, v in good.items() if k != "Product_MRP"} | {"Store_Age_Years": "sixteen"}

frame, errs = validate_payload(good)
assert not errs
print("valid payload ->", {"Sales": round(float(final.predict(frame)[0]), 2)})
frame, errs = validate_payload(bad)
assert frame is None and len(errs) == 2
print("invalid payload -> HTTP 422", errs)

valid payload -> {'Sales': 2882.43}
invalid payload -> HTTP 422 ['missing field: Product_MRP', 'Store_Age_Years: expected int, got str']
